# SS extension experiments — census, calibration, and the cycle walk

Summary of the 2026-07-05 experiment session that shaped the factory
(see `README.md` for the narrative and commit pointers).  Everything here
re-derives from the two census JSONs in this folder; re-run `census.py`
(~3 min, dry — no Velu) after any search change.


In [ ]:
import json
from collections import Counter
import matplotlib.pyplot as plt

mx = json.load(open('census_maxcost.json'))   # legacy cost model (max eigenline degree)
wk = json.load(open('census_walkcost.json'))  # walk cost model (min eigenline degree)
len(mx), len(wk)


## Census headlines

All 856 primes in (1024, 8192]: the rigid-set search succeeds everywhere and
every chosen l-set is executable by the (fixed) SS signature side.


In [ ]:
for name, rows in (('legacy', mx), ('walk', wk)):
    print(name, ' cases:', dict(Counter(r['case'] for r in rows)),
          ' descriptor kinds:', dict(Counter(tuple(r['kinds']) for r in rows)))


## Eigenline degree distribution, before / after the walk

The cost of a generator is the extension degree k of the eigenline the data
builder works in.  The both-direction builder pays the MAX of the two
degrees; the one-direction cycle walk pays the MIN.  The degree pairs are
always {m, 2m} (m odd) or {2m, 2m} (m even) — eigenvalues are ±λ — so the
walk collapses the k=10..18 tail entirely (max anywhere: 9), and 660 primes
get a degree-1 (rational eigenline) generator.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, rows, title in ((axes[0], mx, 'legacy cost = max eigenline degree'),
                        (axes[1], wk, 'walk cost = min eigenline degree')):
    hist = Counter(r['max_k'] for r in rows)
    ks = sorted(hist)
    ax.bar([str(k) for k in ks], [hist[k] for k in ks], color='#4878a8')
    ax.set_yscale('log'); ax.set_title(title, fontsize=11)
    ax.set_xlabel('worst generator eigenline degree k')
    for i, k in enumerate(ks):
        ax.text(i, hist[k], str(hist[k]), ha='center', va='bottom', fontsize=8)
axes[0].set_ylabel('primes (log scale)')
fig.tight_layout()


## Cost calibration (measured, warm single Velu isogeny)

| eigenline degree k | 4 | 6 | 8 | 10 | 12 | 14 |
|---|---|---|---|---|---|---|
| seconds / isogeny | ~0.05 | 2–3 | ~6 | 9–73 | ~16 | ~224 |

The degree dominates; l matters little below ~1000 (a k=6 isogeny costs
2–3 s even at l = 907).  A full graph is |class| isogenies (walk) or
2·|class| (both directions), which is what made the k≥10 tail hours-scale
before the walk.


## The worst primes of the range, before / after

Full-bijection wall clock (both-direction numbers ≥ k=10 are projections
from the per-isogeny calibration; walk numbers are measured):

| p | legacy set | legacy time | walk set | walk time |
|---|---|---|---|---|
| 4013 | (907,) k=6 | 50 s (measured) | (907,) walking k=3 | 8.8 s |
| 6637 | (2713, 3319) k=12 | ~12 min | (71,) k=8 | 7.5 s |
| 8101 | (1051,) k=10 | ~1.5 h | (1051,) walking k=5 | 44 s |
| 6553 | (211,) k=14 | ~4.5 h | (211,) walking k=7 | 31 s |


In [ ]:
for name, rows in (('legacy', mx), ('walk', wk)):
    worst = sorted(rows, key=lambda r: -r['proxy'])[:8]
    print(f'--- {name}: top 8 by cost proxy (order * sum l*k^2)')
    for r in worst:
        print(f"  p={r['p']:>5} {r['case']} h={r['order']:>3}  gens={r['gen_costs']}  ls={r['ls_rig']}")


## Validation coverage

Every prime keeps at least 3 UNUSED split primes with a classical Φ_l on
disk, so the independent edge check in the battery
(`ss_bij_cache.validate_entry`) is available everywhere.


In [ ]:
cov = Counter(r['n_split_unused'] for r in wk)
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ks = sorted(cov)
ax.bar([str(k) for k in ks], [cov[k] for k in ks], color='#7aa66e')
ax.set_xlabel('unused split primes with a cached classical Phi_l')
ax.set_ylabel('primes'); print('min coverage:', min(ks))


## Run estimate

Crude per-prime prediction (calibrated seconds/isogeny × class number ×
generators): **~1.4 h** for the full 856-prime run, dominated by the many
cheap k=1 primes rather than any spike.  Worst single prime measured: 31 s.


In [ ]:
sec = {1: .01, 2: .02, 3: .2, 4: .15, 5: 1.2, 6: .5, 7: .9, 8: .4, 9: 2}
tot = sum(sum(r['order'] * sec.get(k, 10) for l, k in r['gen_costs']) + 2 for r in wk)
print(f'estimated total: {tot/3600:.1f} h')
